# Appendix figures — Cue 1 vs Cue 2 population-vector correlation

Per-session population-vector (PV) correlation between Cue 1 and Cue 2 trials
across track position (the Sun et al. 2025 decorrelation analogue), for all
A6 sessions meeting the thesis inclusion criteria. Three region versions are
produced: **HP** (hippocampus), **mPFC**, and **HP-mPFC** (all units).

**Why this differs from the `PVCueCorr` / dash plot:** the population vectors
are correlated **after per-neuron z-scoring within session** (as in
Sun et al. 2025). Correlating *raw* firing rates is dominated by the
across-neuron mean firing-rate profile and forces all correlations close to 1;
z-scoring removes that and reveals the true correlation structure. This
normalisation is the only substantive change — the 5-bin spatial averaging and
sigma=1 smoothing are unchanged; the colormap, axis track strips and reversal
marker are presentational.

Outputs are written to the thesis repo (`figures/plots/appendix_cuecorr_*.pdf`)
and previewed inline.

In [ ]:
import os, sys, glob
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
from baseVR.base_functionality import init_import_paths
init_import_paths()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.ndimage import gaussian_filter

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

Logger().init_logger(None, None, logging_level='ERROR')
%matplotlib inline

In [ ]:
CUE_INTRO = '2024-11-27'
REVERSAL_SID = '2025-01-23_16-48'            # first session after the reversal
EXCLUDED_SESSIONS = {'2024-11-29_17-21', '2025-01-22_17-51', '2025-01-21_18-49'}
SPATIAL_GROUP = 5                            # 5-bin spatial averaging (as PVCueCorr)
SMOOTH = True

NAS = '/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100'
THESIS_FIG_DIR = '/home/samed/VirtualReality/thesis_repo/figures/plots'
PREVIEW_DIR = os.path.join(os.getcwd(), 'figures_thesis_sam')
os.makedirs(PREVIEW_DIR, exist_ok=True)

# region -> unit-column index slice (units ordered by region; first 20 = HPC)
REGIONS = {'HP': slice(0, 20), 'mPFC': slice(20, None), 'HP-mPFC': slice(None)}
PRETTY = {'HP': 'hippocampus', 'mPFC': 'mPFC', 'HP-mPFC': 'all units, HPC + mPFC'}

# Sun et al. (2025) style correlation colormap (teal=decorrelated, black=0,
# orange=correlated); full r in [-1, 1].
SUN_CMAP = LinearSegmentedColormap.from_list('sun_corr', [
    (0.00, '#46b7d3'), (0.20, '#1b7895'), (0.42, '#07505f'),
    (0.48, '#020202'), (0.52, '#020202'),     # slightly wider black band at r~0
    (0.58, '#7a3a08'), (0.80, '#bd6b17'), (1.00, '#f2a340'),
])
VMIN, VMAX = -1.0, 1.0

# task colours / zones (match the dash track illustration)
CUE1_COL, CUE2_COL = '#e98800', '#bc00e9'
R1_COL, R2_COL = '#787878', '#bebebe'
REWARDED_EDGE = '#40CA72'
REVERSAL_COL = '#d62728'
CUE_ZONE, R1_ZONE, R2_ZONE = (-80.0, 10.0), (50.0, 110.0), (170.0, 230.0)
CUE_VISIBLE = -130.0          # cue 2 becomes visible (track_details cue2_visible start)
# dashed guides: cue visibility + region starts (cue zone, reward 1, reward 2)
BOUNDARY_LINES = [CUE_VISIBLE, CUE_ZONE[0], R1_ZONE[0], R2_ZONE[0]]
min_track, max_track = -169, 261

# zones of interest highlighted as small boxes on the matrix diagonal (each
# zone compared against itself across cue 1 / cue 2), as in Sun et al. 2025.
ZONE_BOXES = [(CUE_ZONE, 'Cue zone'), (R1_ZONE, 'Reward 1 zone'), (R2_ZONE, 'Reward 2 zone')]
ZONE_BOX_COL = '#ffffff'

def draw_track_strip(axs, orient, cue_col, rewarded):
    span = axs.axvspan if orient == 'h' else axs.axhspan
    span(*CUE_ZONE, facecolor=cue_col, lw=0)
    for (z0, z1), rcol, ridx in ((R1_ZONE, R1_COL, 1), (R2_ZONE, R2_COL, 2)):
        if ridx == rewarded:
            span(z0, z1, facecolor=rcol, edgecolor=REWARDED_EDGE, lw=1.3)
        else:
            span(z0, z1, facecolor=rcol, lw=0)
    if orient == 'h':
        axs.set_ylim(0, 1); axs.set_xlim(min_track, max_track)
    else:
        axs.set_xlim(0, 1); axs.set_ylim(max_track, min_track)
    axs.set_xticks([]); axs.set_yticks([])
    for sp in axs.spines.values():
        sp.set_linewidth(0.5)

def draw_zone_boxes(ax):
    '''Outline the three zones of interest as boxes on the matrix diagonal
    (cue-x-cue, R1-x-R1, R2-x-R2), Sun et al. 2025 style.'''
    from matplotlib.patches import Rectangle
    for (z0, z1), _label in ZONE_BOXES:
        ax.add_patch(Rectangle((z0, z0), z1 - z0, z1 - z0, fill=False,
                               edgecolor=ZONE_BOX_COL, lw=1.6, zorder=5))

In [ ]:
# post cue-introduction, ephys recorded, duration >= 10 min, minus the three
# quality/length-excluded sessions used elsewhere in the thesis.
meta = analytics.get_analytics('SessionMetadata', mode='set', paradigm_ids=[1100], animal_ids=[6])
meta_by_sess = meta.groupby(level='session_id').first()
included = []
for sid in sorted(meta_by_sess.index):
    mr = meta_by_sess.loc[sid]
    if (sid[:10] >= CUE_INTRO) and bool(mr['ephys_traces_recorded']) \
            and float(mr['duration_minutes']) >= 10 and sid not in EXCLUDED_SESSIONS:
        if glob.glob(f'{NAS}/{sid}*/session_analytics/FiringRateTrackwiseHz.parquet'):
            included.append(sid)
print(f"Included {len(included)} sessions")
included

In [ ]:
def load_session_pvs(sid):
    '''PVs frame (index: from_position_bin, cue; cols: Unit0001..).'''
    fp = glob.glob(f'{NAS}/{sid}*/session_analytics/FiringRateTrackwiseHz.parquet')[0]
    df = pd.read_parquet(fp)
    df = df.set_index(['trial_id', 'from_position_bin', 'cue', 'trial_outcome',
                       'choice_R1', 'choice_R2'])
    units = sorted(c for c in df.columns if str(c).startswith('Unit'))
    fr = df[units].fillna(0.0)
    return fr.groupby(level=('from_position_bin', 'cue')).mean()

def zscored_corr(PVs_region):
    '''Per-neuron z-score (within session), then cue1 x cue2 PV correlation.
    The z-score step is THE change vs. the raw PVCueCorr analytic.'''
    Z = PVs_region.copy()
    mu, sd = Z.mean(0), Z.std(0).replace(0, 1.0)
    Z = (Z - mu) / sd                                # per-neuron z-score
    pv1, pv2 = Z.xs(1, level='cue'), Z.xs(2, level='cue')
    idx = pv1.index.intersection(pv2.index)
    pv1, pv2 = pv1.loc[idx], pv2.loc[idx]
    a1, a2, pos = [], [], []
    for i in range(0, len(idx), SPATIAL_GROUP):
        a1.append(pv1.iloc[i:i + SPATIAL_GROUP].mean(0))
        a2.append(pv2.iloc[i:i + SPATIAL_GROUP].mean(0))
        pos.append(idx[i])
    A1 = pd.concat(a1, axis=1).T.to_numpy(); A2 = pd.concat(a2, axis=1).T.to_numpy()
    n = len(pos); C = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            xi, yj = A1[i], A2[j]
            if np.std(xi) > 0 and np.std(yj) > 0:
                C[i, j] = np.corrcoef(xi, yj)[0, 1]
    pos = np.asarray(pos, dtype=float)
    if SMOOTH:
        C = gaussian_filter(np.nan_to_num(C, nan=0.0), sigma=1)
    return pos, C

print("Loading + computing z-scored PV correlations ...")
SESSION_PVS = {sid: load_session_pvs(sid) for sid in included}
print("done")

In [ ]:
def make_figure(region, save=True, sessions=None, session_pvs=None,
                stem_prefix='appendix_cuecorr_', title_note=''):
    sessions = included if sessions is None else sessions
    session_pvs = SESSION_PVS if session_pvs is None else session_pvs
    sl = REGIONS[region]
    plt.rcParams.update({
        'font.size': 15, 'axes.labelsize': 20,
        'xtick.labelsize': 12, 'ytick.labelsize': 12, 'font.family': 'sans-serif',
    })
    ncols = 5
    nrows = int(np.ceil(len(sessions) / ncols))
    figh = nrows * 3.7
    fig = plt.figure(figsize=(ncols * 3.7 + 1.4, figh))
    gs = GridSpec(nrows, ncols + 1, figure=fig,
                  width_ratios=[1] * ncols + [0.07], wspace=0.22, hspace=0.42)
    im = None
    for k, sid in enumerate(sessions):
        r, c = divmod(k, ncols)
        ax = fig.add_subplot(gs[r, c])
        pos, C = zscored_corr(session_pvs[sid].iloc[:, sl])
        im = ax.imshow(C, origin='upper', aspect='auto', cmap=SUN_CMAP, vmin=VMIN, vmax=VMAX,
                       extent=[pos.min(), pos.max(), pos.max(), pos.min()])
        ax.plot([max(min_track, pos.min()), min(max_track, pos.max())],
                [max(min_track, pos.min()), min(max_track, pos.max())],
                color='white', lw=0.7, ls=':', alpha=0.6)
        # dashed guides for cue visibility and region starts (both axes)
        for b in BOUNDARY_LINES:
            ax.axvline(b, color='white', ls='--', lw=0.6, alpha=0.5)
            ax.axhline(b, color='white', ls='--', lw=0.6, alpha=0.5)
        # three zones of interest as boxes on the diagonal (Sun et al. 2025)
        draw_zone_boxes(ax)
        ax.set_xlim(min_track, max_track); ax.set_ylim(max_track, min_track)
        ax.set_yticks([])
        ax.set_xticks([-100, 0, 100, 200] if r == nrows - 1 else [])
        ax.tick_params(length=3)

        is_rev = (sid == REVERSAL_SID)
        if is_rev:
            for sp in ax.spines.values():
                sp.set_edgecolor(REVERSAL_COL); sp.set_linewidth(2.4)

        divider = make_axes_locatable(ax)
        ax_top = divider.append_axes('top', size='7%', pad=0.04, sharex=ax)
        ax_left = divider.append_axes('left', size='7%', pad=0.04, sharey=ax)
        draw_track_strip(ax_top, 'h', CUE2_COL, rewarded=2)
        draw_track_strip(ax_left, 'v', CUE1_COL, rewarded=1)
        _d, _t = sid.split('_')
        ttl = f"{_d}  {_t.replace('-', ':')}"
        if is_rev:
            ax_top.set_title('↺ reversal\n' + ttl, fontsize=12.5, pad=3,
                             color=REVERSAL_COL, fontweight='bold')
        else:
            ax_top.set_title(ttl, fontsize=12.5, pad=3)

    for k in range(len(sessions), nrows * ncols):
        r, c = divmod(k, ncols)
        fig.add_subplot(gs[r, c]).axis('off')

    cax = fig.add_subplot(gs[:, ncols])
    cb = fig.colorbar(im, cax=cax, ticks=[-1, -0.5, 0, 0.5, 1])
    cb.set_label('Population-vector correlation (Pearson $r$)', fontsize=18)

    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    handles = [
        Patch(facecolor=CUE1_COL, label='Cue 1'),
        Patch(facecolor=CUE2_COL, label='Cue 2'),
        Patch(facecolor=R2_COL, edgecolor='0.4', label='Reward zone'),
        Patch(facecolor='white', edgecolor=REWARDED_EDGE, lw=1.6, label='Rewarded zone'),
        Patch(facecolor='none', edgecolor=ZONE_BOX_COL, lw=1.6, label='Zone of interest'),
        Line2D([0], [0], color=REVERSAL_COL, lw=2.4, label='Reversal session'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=6, fontsize=15,
               frameon=False, bbox_to_anchor=(0.5, -0.005))
    fig.suptitle(f'Cue 1 vs Cue 2 population-vector correlation across track position '
                 f'({PRETTY[region]}), per session{title_note}\n'
                 f'per-neuron z-scored within session', fontsize=20, y=1 + 0.06 / figh)
    fig.supxlabel('Cue 2 trials — track position [cm]', fontsize=20, y=0.444 / figh)
    fig.supylabel('Cue 1 trials — track position [cm]', fontsize=20, x=0.008)
    # margins in fixed physical inches so the layout holds for any row count
    fig.subplots_adjust(left=0.06, right=0.93, top=1 - 0.888 / figh, bottom=1.184 / figh)

    if save:
        stem = stem_prefix + region.replace('-', '_')
        fig.savefig(os.path.join(THESIS_FIG_DIR, stem + '.pdf'), bbox_inches='tight', dpi=300)
        fig.savefig(os.path.join(PREVIEW_DIR, stem + '.svg'), bbox_inches='tight', dpi=200)
    return fig

## All units (HPC + mPFC)

In [ ]:
# all units (HPC + mPFC) -- this is the figure wired to \\Cref{Appendix}
make_figure('HP-mPFC');

## Hippocampus

In [ ]:
# hippocampus
make_figure('HP');

## mPFC

In [ ]:
# mPFC
make_figure('mPFC');

## Sessions before cue introduction

Same z-scored Cue 1 vs Cue 2 population-vector correlation, but for the
sessions *before* the cue was introduced (`< 2024-11-27`). Same ephys /
duration inclusion criteria, three region versions as above.

In [ ]:
# --------- pre-cue-introduction sessions (same criteria, date < CUE_INTRO) --
included_pre = []
for sid in sorted(meta_by_sess.index):
    mr = meta_by_sess.loc[sid]
    if (sid[:10] < CUE_INTRO) and bool(mr['ephys_traces_recorded']) \
            and float(mr['duration_minutes']) >= 10 and sid not in EXCLUDED_SESSIONS:
        if glob.glob(f'{NAS}/{sid}*/session_analytics/FiringRateTrackwiseHz.parquet'):
            included_pre.append(sid)
print(f"Included {len(included_pre)} pre-cue sessions")

SESSION_PVS_PRE = {sid: load_session_pvs(sid) for sid in included_pre}

# all units (HPC + mPFC); also produces HP / mPFC versions below
make_figure('HP-mPFC', sessions=included_pre, session_pvs=SESSION_PVS_PRE,
            stem_prefix='appendix_cuecorr_precue_', title_note=', before cue introduction');
make_figure('HP', sessions=included_pre, session_pvs=SESSION_PVS_PRE,
            stem_prefix='appendix_cuecorr_precue_', title_note=', before cue introduction');
make_figure('mPFC', sessions=included_pre, session_pvs=SESSION_PVS_PRE,
            stem_prefix='appendix_cuecorr_precue_', title_note=', before cue introduction');

# Joint plates — all sessions per region (pre-cue + post-introduction)

A single full-page plate per region (**HP-mPFC**, **HP**, **mPFC**) showing
*every* recorded session, grouped into two labelled blocks: the **pre-cue
baseline** (before 27 Nov 2024) on top and the **post-introduction** sessions
(ordered over learning) below. These are the versions wired into the thesis
appendix and **replace** the per-phase plates above (same canonical filenames
`appendix_cuecorr_{HP_mPFC,HP,mPFC}.pdf`). The per-phase cells above are kept
for reference and are left unchanged.

In [ ]:
# ------- JOINT plates: all sessions per region (pre-cue + post-introduction) -------
# One full page per region with a uniform SQUARE-panel grid: a pre-cue baseline
# block (Block A, top row) above the post-introduction sessions (Block B). All
# heatmaps share one quadratic format (6 columns throughout); the colourbar and
# legend fill the empty cells of the last Block-B row. The figure is authored
# near \textwidth so panel text stays legible in print. These REPLACE the
# per-phase plates in the appendix (same canonical filenames). Reuses the helpers
# / PVs defined above; the per-phase cells are left untouched. Mirrors
# make_appendix_cuecorr_figure.py.
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

POS_TICKS = [-100, 0, 100, 200]
# font sizes (figure authored near final print width -> roughly WYSIWYG)
FS_TITLE, FS_TICK, FS_SUPLAB = 9, 7.5, 12
FS_HEAD, FS_SUP, FS_CBAR, FS_LEG = 12, 13.5, 9.5, 9


def _draw_panel(ax, sid, pos, C, show_xticks, show_yticks):
    im = ax.imshow(C, origin='upper', aspect='auto', cmap=SUN_CMAP, vmin=VMIN, vmax=VMAX,
                   extent=[pos.min(), pos.max(), pos.max(), pos.min()])
    lo, hi = max(min_track, pos.min()), min(max_track, pos.max())
    ax.plot([lo, hi], [lo, hi], color='white', lw=0.7, ls=':', alpha=0.6)
    for b in BOUNDARY_LINES:
        ax.axvline(b, color='white', ls='--', lw=0.5, alpha=0.5)
        ax.axhline(b, color='white', ls='--', lw=0.5, alpha=0.5)
    draw_zone_boxes(ax)
    ax.set_xlim(min_track, max_track); ax.set_ylim(max_track, min_track)
    ax.set_yticks([])
    ax.tick_params(length=2.5, labelsize=FS_TICK)
    if show_xticks:                   # 45deg so all four values stay legible in a ~22mm panel
        ax.set_xticks(POS_TICKS)
        ax.set_xticklabels([str(t) for t in POS_TICKS], rotation=45, ha='right',
                           rotation_mode='anchor', fontsize=FS_TICK)
        ax.tick_params(axis='x', pad=1)
    else:
        ax.set_xticks([])
    if sid == REVERSAL_SID:
        for sp in ax.spines.values():
            sp.set_edgecolor(REVERSAL_COL); sp.set_linewidth(2.2)
    # track strips: no sharex/sharey (a shared axis would let the strip's empty
    # ticks wipe the panel's position ticks); they align via explicit limits.
    divider = make_axes_locatable(ax)
    ax_top = divider.append_axes('top', size='7%', pad=0.04)
    ax_left = divider.append_axes('left', size='7%', pad=0.04)
    draw_track_strip(ax_top, 'h', CUE2_COL, rewarded=2)
    draw_track_strip(ax_left, 'v', CUE1_COL, rewarded=1)
    if show_yticks:                   # y position numbers sit on the left strip
        ax_left.set_yticks(POS_TICKS)
        ax_left.set_yticklabels([str(t) for t in POS_TICKS])
        ax_left.tick_params(axis='y', length=2.5, labelsize=FS_TICK, left=True, labelleft=True)
    d, t = sid.split('_'); t = t.replace('-', ':')
    if sid == REVERSAL_SID:
        ax_top.set_title(f'↺ reversal\n{d}  {t}', fontsize=FS_TITLE, pad=2.5,
                         color=REVERSAL_COL, fontweight='bold', linespacing=1.1)
    else:
        ax_top.set_title(f'{d}\n{t}', fontsize=FS_TITLE, pad=2.5, linespacing=1.1)
    return im


def _legend_handles():
    return [
        Patch(facecolor=CUE1_COL, label='Cue 1 (rows)'),
        Patch(facecolor=CUE2_COL, label='Cue 2 (columns)'),
        Patch(facecolor=R2_COL, edgecolor='0.4', label='Reward zone'),
        Patch(facecolor='white', edgecolor=REWARDED_EDGE, lw=1.4, label='Rewarded zone'),
        Patch(facecolor='none', edgecolor='0.2', lw=1.4, label='Zone of interest'),
        Line2D([0], [0], color=REVERSAL_COL, lw=2.2, label='Reversal session'),
    ]


def make_joint_figure(region, save=True):
    """Full-page joint plate with uniform square panels: pre-cue baseline
    (Block A, one row) above the post-introduction sessions (Block B), one
    region. 6 columns throughout -> every heatmap is the same quadratic size."""
    sl = REGIONS[region]
    plt.rcParams.update({'font.family': 'sans-serif'})
    pre, post = included_pre, included
    ncols = 6
    nA = 1
    nB = int(np.ceil(len(post) / ncols))
    wspace, hspace = 0.20, 0.50
    left, right = 0.085, 0.965
    W = 6.8
    cell_w_f = (right - left) / (ncols + (ncols - 1) * wspace); cell_w_in = cell_w_f * W
    # square cells: derive the vertical budget (inches) from the cell width
    top_in, A_header_in, gapAB_in, bottom_in = 0.72, 0.62, 0.82, 0.58
    A_band_in = cell_w_in * nA
    B_band_in = cell_w_in * (nB + (nB - 1) * hspace)
    H = top_in + A_header_in + A_band_in + gapAB_in + B_band_in + bottom_in
    A_top = 1 - (top_in + A_header_in) / H; A_bot = A_top - A_band_in / H
    B_top = A_bot - gapAB_in / H;          B_bot = B_top - B_band_in / H
    title_h = 0.27 / H                # vertical room a 2-line panel title occupies

    fig = plt.figure(figsize=(W, H))
    gsA = fig.add_gridspec(nA, ncols, left=left, right=right, top=A_top, bottom=A_bot, wspace=wspace)
    gsB = fig.add_gridspec(nB, ncols, left=left, right=right, top=B_top, bottom=B_bot,
                           wspace=wspace, hspace=hspace)

    last_row_of_col = {}             # bottom-most filled Block-B row per column (for x-numbers)
    for k in range(len(post)):
        r, c = divmod(k, ncols); last_row_of_col[c] = max(last_row_of_col.get(c, -1), r)

    im = None
    for k, sid in enumerate(pre):
        ax = fig.add_subplot(gsA[0, k])
        pos, C = zscored_corr(SESSION_PVS_PRE[sid].iloc[:, sl])
        im = _draw_panel(ax, sid, pos, C, show_xticks=False, show_yticks=(k == 0))
    for k, sid in enumerate(post):
        r, c = divmod(k, ncols); ax = fig.add_subplot(gsB[r, c])
        pos, C = zscored_corr(SESSION_PVS[sid].iloc[:, sl])
        im = _draw_panel(ax, sid, pos, C, show_xticks=(r == last_row_of_col[c]), show_yticks=(c == 0))

    # locate the empty cells of the last Block-B row -> colourbar + legend
    tw = right - left; cwf = tw / (ncols + (ncols - 1) * wspace); gwf = wspace * cwf
    th = B_top - B_bot; chf = th / (nB + (nB - 1) * hspace); ghf = hspace * chf
    colx = lambda c: (left + c * (cwf + gwf), left + c * (cwf + gwf) + cwf)
    rowy = lambda r: (B_top - r * (chf + ghf) - chf, B_top - r * (chf + ghf))
    empty_cols = [c for c in range(ncols) if last_row_of_col.get(c, -1) < nB - 1]
    if empty_cols:
        ex0 = colx(empty_cols[0])[0]; ex1 = colx(empty_cols[-1])[1]
        ey0, ey1 = rowy(nB - 1)
        cb_w = (ex1 - ex0) * 0.86; cb_x = ex0 + ((ex1 - ex0) - cb_w) / 2
        cax = fig.add_axes([cb_x, ey1 - 0.018, cb_w, 0.013])
        cb = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=[-1, -0.5, 0, 0.5, 1])
        cb.set_label('Population-vector correlation (Pearson $r$)', fontsize=FS_CBAR, labelpad=2)
        cb.ax.tick_params(labelsize=FS_TICK, length=2, pad=1)
        cb.ax.xaxis.set_ticks_position('top'); cb.ax.xaxis.set_label_position('bottom')
        fig.legend(handles=_legend_handles(), loc='center', ncol=2, fontsize=FS_LEG, frameon=False,
                   bbox_to_anchor=((ex0 + ex1) / 2, (ey0 + ey1 - 0.04) / 2 - 0.005),
                   handlelength=1.4, columnspacing=1.2, labelspacing=0.5)
    else:                            # fallback: side colourbar + bottom legend
        cax = fig.add_axes([right + 0.02, B_bot, 0.016, A_top - B_bot])
        cb = fig.colorbar(im, cax=cax, ticks=[-1, -0.5, 0, 0.5, 1])
        cb.set_label('Population-vector correlation (Pearson $r$)', fontsize=FS_CBAR)
        fig.legend(handles=_legend_handles(), loc='lower center', ncol=6, fontsize=FS_LEG,
                   frameon=False, bbox_to_anchor=(0.5, 0.01))

    mid = (left + right) / 2
    A_head_y = A_top + title_h + 0.018      # headers sit ABOVE their first row's panel titles
    B_head_y = B_top + title_h + 0.014
    fig.text(mid, A_head_y, 'Before cue introduction  ·  pre-cue baseline',
             ha='center', va='bottom', fontsize=FS_HEAD, fontweight='bold')
    yd = (A_bot + B_head_y) / 2             # divider between Block A panels and the Block B header
    fig.add_artist(Line2D([left, right], [yd, yd], color='0.6', lw=0.9, transform=fig.transFigure))
    fig.text(mid, B_head_y, 'After cue introduction  ·  learning →',
             ha='center', va='bottom', fontsize=FS_HEAD, fontweight='bold')
    fig.suptitle(f'Cue 1 vs Cue 2 population-vector correlation across track position '
                 f'({PRETTY[region]})\nall sessions · per-neuron z-scored within session',
                 fontsize=FS_SUP, y=1 - 0.30 / H)
    fig.supxlabel('Cue 2 trials — track position [cm]', fontsize=FS_SUPLAB, y=0.012)
    fig.supylabel('Cue 1 trials — track position [cm]', fontsize=FS_SUPLAB, x=0.012)

    if save:
        stem = f'appendix_cuecorr_{region.replace("-", "_")}'
        fig.savefig(os.path.join(THESIS_FIG_DIR, stem + '.pdf'), bbox_inches='tight', dpi=300)
        fig.savefig(os.path.join(PREVIEW_DIR, stem + '.svg'), bbox_inches='tight', dpi=200)
    return fig


# joint plates wired to the appendix (these replace the per-phase plates there)
make_joint_figure('HP-mPFC')
make_joint_figure('HP')
make_joint_figure('mPFC');

# Second appendix figure — all 23 ICA ensembles

Projections and single-neuron weights for the **full set of 23 ICA ensembles**
(wired to `\Cref{Appendix_trackwise_ensembles}`). Generalises the two analysed
ensembles shown in the methods/results to all 23. **Redesigned for legibility**
(each panel far taller in print than the old single landscape page):

- **Projections** (`appendix_ensemble_projections.pdf`, **one A4 page, two
  columns** — ensembles 1–12 left, 13–23 right): per ensemble, the projection
  across the concatenated recording, smoothed at 15 min (solid) and 30 s
  (faint); session boundaries dashed, reversal (2025-01-23) marked red. Ensemble
  names sit inside each panel (top-left) so the two columns never overlap.
- **Weights** (`appendix_ensemble_weights.pdf`, **two columns** on one page):
  per ensemble, the per-neuron ICA weight vector (HP 1–20 yellow, mPFC 21–77
  magenta; rings on `|weight| > 0.2`); ensemble names inside each panel.

Mirrors `make_appendix_ensembles_figure.py`. Outputs are written to the thesis
repo and previewed inline.

In [ ]:
# --------- load animal-level ICA ensembles (weights + concatenated proj) -----
# Same inclusion as the rest of the A6 analyses: all sessions minus the three
# quality/length-excluded ones. The ensembles are animal-level analytics
# computed over the full concatenated recording.
from analytics_processing.sessions_from_nas_parsing import (
    sessionlist_fullfnames_from_args, fullfnames2snames,
)

EXCLUDED_SESSION_NAMES = [
    '2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min',
    '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min',
    '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min',
]
_ens_session_dirs = sessionlist_fullfnames_from_args(
    [1100], [6], None, excl_session_names=EXCLUDED_SESSION_NAMES)[0]
ens_session_names = fullfnames2snames(_ens_session_dirs)

ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=ens_session_names)
ensamble_proj = analytics.get_analytics(
    'ConcatenatedEnsambleProj40ms', session_names=ens_session_names).drop('to_ephys_timestamp', axis=1)
ensamble_proj.set_index(['session_id', 'from_ephys_timestamp'], inplace=True)

N_UNITS, N_ENS = ensambles.shape
print(f'{N_ENS} ensembles, {N_UNITS} units, '
      f'{ensamble_proj.index.get_level_values("session_id").nunique()} sessions')

In [ ]:
# ----- second appendix figure, page 1: ensemble projections (ONE A4 page, two columns) -----
# Readability redesign: all 23 ensembles on a single portrait page in two
# columns (12 + 11), so each panel is far taller in print than the old single
# landscape page while still fitting one A4 page. 15 min mean (solid) + 30 s mean
# (faint), clipped to +/-3 before smoothing; session boundaries dashed; reversal
# red. Ensemble names sit INSIDE each panel (top-left) to avoid any label/plot
# overlap between the two columns. Mirrors make_appendix_ensembles_figure.py.
from matplotlib.lines import Line2D

PROJ_CLIP = 3
WIN_15MIN = 25 * 60 * 15          # 15 min rolling window (25 Hz -> 40 ms bins)
MINP_15MIN = 25 * 60 * 8
WIN_30S = 25 * 30                 # 30 s rolling window
REVERSAL_DATE = '2025-01-23'
PROJ_15MIN_COL, PROJ_30S_COL = '#1f4e9c', '#6a9fd8'
# authored wider than \textwidth -> author fonts ~1.5x the print target
FS_ENS, FS_DATE, FS_SUPLAB, FS_SUP, FS_LEG = 10.5, 8, 13, 16, 11


def _ens_label(ax, name, fs=FS_ENS):
    ax.text(0.010, 0.84, name, transform=ax.transAxes, fontsize=fs, fontweight='bold',
            va='center', ha='left',
            bbox=dict(boxstyle='round,pad=0.14', fc='white', ec='none', alpha=0.72))


def _proj_geometry():
    session_ids = list(ensamble_proj.index.unique(level='session_id'))
    boundaries, date_labels, cnt, rev_x = [], [], 0, None
    for s_id in session_ids:
        boundaries.append(cnt)
        date = str(s_id).split('_')[0]; date_labels.append(date)
        if rev_x is None and date >= REVERSAL_DATE:
            rev_x = cnt
        cnt += len(ensamble_proj.loc[s_id])
    return boundaries, date_labels, rev_x, cnt


def make_projection_figure():
    boundaries, date_labels, rev_x, total_len = _proj_geometry()
    date_short = [d[5:] for d in date_labels]      # month-day (year obvious)
    cols = list(ensamble_proj.columns)
    W, H = 8.0, 10.2
    ncol = 2
    nrow = int(np.ceil(N_ENS / ncol))              # 12 (12 + 11); last right cell empty
    fig = plt.figure(figsize=(W, H))
    gs = fig.add_gridspec(nrow, ncol, left=0.075, right=0.985, top=0.918, bottom=0.085,
                          wspace=0.09, hspace=0.0)
    col_counts = [min(nrow, N_ENS), N_ENS - min(nrow, N_ENS)]
    for idx, col in enumerate(cols):
        c = idx // nrow; r = idx % nrow
        is_bottom = (r == col_counts[c] - 1)
        ax = fig.add_subplot(gs[r, c])
        clipped = np.clip(ensamble_proj[col], -PROJ_CLIP, PROJ_CLIP)
        s30 = clipped.groupby(level='session_id').rolling(
            window=WIN_30S, center=True, min_periods=1).mean().values
        s15 = clipped.groupby(level='session_id').rolling(
            window=WIN_15MIN, center=True, min_periods=MINP_15MIN).mean().values
        ax.plot(s30, color=PROJ_30S_COL, alpha=0.5, linewidth=0.5)
        ax.plot(s15, color=PROJ_15MIN_COL, alpha=0.95, linewidth=1.2)
        for b in boundaries:
            ax.axvline(b, color='gray', linestyle='--', linewidth=0.4, alpha=0.4)
        if rev_x is not None:
            ax.axvline(rev_x, color=REVERSAL_COL, linewidth=1.0, alpha=0.9)
        ax.set_xlim(0, total_len)
        ax.set_yticks([]); ax.tick_params(axis='y', length=0)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        _ens_label(ax, str(col).replace('Assembly', 'Ens '))
        if is_bottom:
            ax.set_xticks(boundaries)
            ax.set_xticklabels(date_short, rotation=90, fontsize=FS_DATE)
            ax.tick_params(axis='x', labelbottom=True, length=2, pad=1)
        else:
            ax.set_xticks([])
    fig.supylabel('Projection (a.u., per-panel scale)', fontsize=FS_SUPLAB, x=0.020)
    fig.supxlabel('Session  (concatenated 40 ms recording; month-day labels at session starts)',
                  fontsize=FS_SUPLAB, y=0.013)
    fig.suptitle(f'Ensemble projections across the recording  ·  all {N_ENS} ICA ensembles',
                 fontsize=FS_SUP, x=0.075, ha='left', y=0.985)
    handles = [
        Line2D([0], [0], color=PROJ_15MIN_COL, lw=2.2, label='15 min mean'),
        Line2D([0], [0], color=PROJ_30S_COL, alpha=0.6, lw=2.2, label='30 s mean'),
        Line2D([0], [0], color=REVERSAL_COL, lw=2.2, label='Reversal session'),
    ]
    fig.legend(handles=handles, ncol=3, loc='upper center', bbox_to_anchor=(0.55, 0.957),
               fontsize=FS_LEG, frameon=False, handlelength=1.8, columnspacing=2.0)
    fig.savefig(os.path.join(THESIS_FIG_DIR, 'appendix_ensemble_projections.pdf'),
                bbox_inches='tight', dpi=300)
    fig.savefig(os.path.join(PREVIEW_DIR, 'appendix_ensemble_projections.svg'),
                bbox_inches='tight', dpi=150)
    print('wrote appendix_ensemble_projections.pdf (1 page)')
    return fig


make_projection_figure();

In [ ]:
# ----- second appendix figure, page 2: ensemble weights (one page, two columns) -----
# Readability redesign + overlap fix: the 23 ensembles in two columns (12 + 11)
# on one portrait page. The ensemble name is drawn INSIDE each panel (top-left)
# instead of as a left-side y-label, which removes the previous overlaps (right
# column's row labels over the left column's plot, and the y-axis label over the
# row labels). HP units (1-20) yellow, mPFC (21-77) magenta, black rings on
# |weight| > 0.2, shared weight scale. Mirrors make_appendix_ensembles_figure.py.
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

HP_NEURON_COUNT = 20
WEIGHT_THR = 0.2
HP_COL, MPFC_COL = '#e6b800', '#c026c0'
# authored wider than \textwidth -> author fonts ~1.5x the print target
FS_ENS_W, FS_XNUM, FS_SUPLAB_W, FS_SUP_W, FS_LEG_W = 10.5, 10.5, 13, 16, 11


def _ens_label_w(ax, name, fs=FS_ENS_W):
    ax.text(0.010, 0.84, name, transform=ax.transAxes, fontsize=fs, fontweight='bold',
            va='center', ha='left',
            bbox=dict(boxstyle='round,pad=0.14', fc='white', ec='none', alpha=0.72))


def make_weights_figure():
    ensemble_cols = list(ensambles.columns)
    ylim = np.max(np.abs(ensambles.values)) * 1.1
    W, H = 8.4, 10.4
    ncol = 2
    nrow = int(np.ceil(N_ENS / ncol))                 # 12 rows (12 + 11)
    fig = plt.figure(figsize=(W, H))
    gs = fig.add_gridspec(nrow, ncol, left=0.075, right=0.985, top=0.918, bottom=0.072,
                          wspace=0.09, hspace=0.0)
    col_counts = [min(nrow, N_ENS), N_ENS - min(nrow, N_ENS)]
    for idx, col in enumerate(ensemble_cols):
        c = idx // nrow
        r = idx % nrow
        is_bottom = (r == col_counts[c] - 1)
        ax = fig.add_subplot(gs[r, c])
        weights = ensambles[col]
        ml1 = ax.stem(np.arange(HP_NEURON_COUNT), weights.iloc[:HP_NEURON_COUNT],
                      linefmt='-', markerfmt='o', basefmt=' ')
        ml2 = ax.stem(np.arange(HP_NEURON_COUNT, N_UNITS), weights.iloc[HP_NEURON_COUNT:],
                      linefmt='-', markerfmt='o', basefmt=' ')
        for ml, cl in ((ml1, HP_COL), (ml2, MPFC_COL)):
            ml.markerline.set_markersize(2.1); ml.markerline.set_color(cl)
            ml.stemlines.set_linewidth(0.55); ml.stemlines.set_color(cl)
        mask = (np.abs(weights) > WEIGHT_THR).values
        ax.scatter(np.arange(N_UNITS)[mask], weights.values[mask], s=13,
                   facecolors='none', edgecolors='k', linewidths=0.7, zorder=3)
        ax.axhline(0, color='k', linewidth=0.4, alpha=0.5)
        ax.axvline(HP_NEURON_COUNT - 0.5, color='gray', linestyle='--', linewidth=0.5, alpha=0.6)
        ax.set_xlim(-1, N_UNITS); ax.set_ylim(-ylim, ylim)
        ax.set_yticks([]); ax.tick_params(axis='y', length=0)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        _ens_label_w(ax, str(col).replace('Assembly', 'Ens '))
        if is_bottom:
            ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x + 1:g}'))
            ax.set_xticks([0, 19, 39, 59, N_UNITS - 1])
            ax.tick_params(axis='x', labelsize=FS_XNUM, length=2, pad=1)
        else:
            ax.set_xticks([])
    fig.supylabel(f'ICA weight (shared scale, ±{ylim:.2f})', fontsize=FS_SUPLAB_W, x=0.020)
    fig.supxlabel('Neuron index  (HP 1–20  |  mPFC 21–77)', fontsize=FS_SUPLAB_W, y=0.012)
    fig.suptitle(f'Single-neuron ICA weights  ·  all {N_ENS} ensembles',
                 fontsize=FS_SUP_W, x=0.075, ha='left', y=0.985)
    handles = [
        Line2D([0], [0], color=HP_COL, marker='o', linestyle='-', markersize=6, label='HP unit'),
        Line2D([0], [0], color=MPFC_COL, marker='o', linestyle='-', markersize=6, label='mPFC unit'),
        Line2D([0], [0], color='k', marker='o', linestyle='none', markerfacecolor='none',
               markeredgecolor='k', markersize=8, label='|weight| > 0.2'),
    ]
    fig.legend(handles=handles, ncol=3, loc='upper center', bbox_to_anchor=(0.55, 0.957),
               fontsize=FS_LEG_W, frameon=False, handlelength=1.8, columnspacing=2.0)
    fig.savefig(os.path.join(THESIS_FIG_DIR, 'appendix_ensemble_weights.pdf'),
                bbox_inches='tight', dpi=300)
    fig.savefig(os.path.join(PREVIEW_DIR, 'appendix_ensemble_weights.svg'),
                bbox_inches='tight', dpi=150)
    print('wrote appendix_ensemble_weights.pdf')
    return fig


make_weights_figure();

# Third appendix figure — single-neuron trackwise activation atlas

Companion to thesis **Figure 4b**, generalised from one example unit to **all
77 single units** and **pooled across cues**. Every panel is a
session × track-position heatmap of one neuron's mean firing rate (all trials,
both cue 1 and cue 2). Unlike Figure 4b there is **no cue split** — each panel
shows the overall trackwise activation of a single neuron over learning, and
carries **its own little track-zone strip** directly above it (cue / reward
zones + cue-visible marker), aligned to that panel's track-position axis.

Panels are **peak-normalised per neuron** so each unit's spatial-tuning pattern
is legible regardless of its absolute rate; the absolute peak (Hz) is printed in
every panel title so the quantitative scale is preserved. Sessions run top
(earliest) → bottom (latest); the cue-introduction (2024-11-27) and reversal
(2025-01-23) boundaries are dashed lines, and faint dotted guides mark the
cue-visible point and the cue / reward-zone onsets within each heatmap. The
whole atlas is laid out on a single **A4 portrait page** (7 columns × 11 rows)
for use as a high-resolution supplementary figure. Session set = every ephys
session ≥ 10 min minus the three quality/length-excluded sessions (26 sessions);
units 1–20 = hippocampus, 21–77 = mPFC.

In [ ]:
# Third appendix figure — single-neuron trackwise activation atlas (all 77 units)
# Companion to thesis Figure 4b, generalised to every single unit and pooled
# across cues. Each small panel is a session x track-position heatmap of one
# neuron's mean firing rate (all trials, both cue 1 and cue 2 -> NO cue split),
# with its OWN little track-zone strip directly above it (cue / reward zones +
# cue-visible marker), exactly like Figure 4b's per-panel schematic. Panels are
# peak-normalised per neuron so each unit's spatial-tuning pattern is legible
# regardless of its absolute rate; the absolute peak (Hz) is printed in every
# panel title so the quantitative scale is preserved. One A4 portrait page,
# 7 cols x 11 rows = 77 panels. Reuses the config-cell constants (NAS, zones,
# colours, REVERSAL_SID, ...). Saved without bbox_inches='tight' so the canvas
# stays exactly A4.
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

# atlas-only style constants (atlas-prefixed so they never clobber the ensemble
# figures' HP_COL / MPFC_COL defined in the cells above)
ATLAS_CUE_NEUTRAL = '#b9a7d6'        # cue zone in the strips (data are cue-pooled -> neutral)
ATLAS_HP_COL, ATLAS_MPFC_COL = '#2f6fba', '#c0561a'
ATLAS_CUEINTRO_COL = '#1b7895'
ATLAS_REV_DATE = REVERSAL_SID[:10]
ATLAS_N_HP = 20                      # first 20 units = hippocampus (see REGIONS)
ATLAS_SMOOTH_SIG = (0.6, 2.0)        # (session, position) gaussian sigma, in bins
ATLAS_CMAP = 'Greys'

def _atlas_nan_gaussian(M, sigma):
    '''NaN-aware 2D gaussian smoothing (normalised convolution).'''
    mask = np.isfinite(M).astype(float)
    num = gaussian_filter(np.where(mask > 0, M, 0.0), sigma=sigma, mode='nearest')
    den = gaussian_filter(mask, sigma=sigma, mode='nearest')
    out = np.divide(num, den, out=np.full_like(num, np.nan), where=den > 1e-6)
    out[mask == 0] = np.nan          # keep genuinely-missing bins blank
    return out

# --- session set: every ephys session, minus the 3 quality/length-excluded ---
_atlas_fps = sorted(glob.glob(f'{NAS}/*/session_analytics/FiringRateTrackwiseHz.parquet'))
def _atlas_sid_of(fp):
    return os.path.basename(os.path.dirname(os.path.dirname(fp)))[:16]
_atlas = sorted(((_atlas_sid_of(fp), fp) for fp in _atlas_fps
                 if _atlas_sid_of(fp) not in EXCLUDED_SESSIONS), key=lambda t: t[0])
atlas_sids = [s for s, _ in _atlas]
print(f'{len(atlas_sids)} sessions in the atlas')

def _atlas_load_trackfr(fp):
    '''Per-position mean firing rate, pooled over all trials & cues.'''
    df = pd.read_parquet(fp)
    us = sorted(c for c in df.columns if str(c).startswith('Unit'))
    df['pos'] = pd.to_numeric(df['from_position_bin'], errors='coerce')
    return df.dropna(subset=['pos']).groupby('pos')[us].mean()

atlas_trackfr = {s: _atlas_load_trackfr(fp) for s, fp in _atlas}
atlas_units = sorted(c for c in next(iter(atlas_trackfr.values())).columns
                     if str(c).startswith('Unit'))
atlas_pos = np.sort(np.unique(np.concatenate(
    [m.index.values for m in atlas_trackfr.values()])))
atlas_pos = atlas_pos[(atlas_pos >= min_track) & (atlas_pos <= max_track)]
print(f'{len(atlas_units)} units, {len(atlas_pos)} position bins '
      f'[{atlas_pos.min():.0f}, {atlas_pos.max():.0f}] cm')

def _atlas_neuron_matrix(unit):
    M = np.vstack([atlas_trackfr[s][unit].reindex(atlas_pos).values for s in atlas_sids])
    return _atlas_nan_gaussian(M, ATLAS_SMOOTH_SIG)

_atlas_dates = [s[:10] for s in atlas_sids]
def _atlas_boundary_y(thresh):
    n = sum(d < thresh for d in _atlas_dates)
    return n if 0 < n < len(atlas_sids) else None
_atlas_cue_y = _atlas_boundary_y(CUE_INTRO)
_atlas_rev_y = _atlas_boundary_y(ATLAS_REV_DATE)

def _atlas_fmt_hz(v):
    return 'n/a' if not np.isfinite(v) else (f'{v:.1f}' if v < 10 else f'{v:.0f}')

def _atlas_track_strip(ax):
    '''Thin per-panel track-zone schematic above the heatmap (cue + reward zones
    and the cue-visible marker), aligned to the panel's track-position axis.'''
    top = make_axes_locatable(ax).append_axes('top', size='17%', pad=0.025)
    top.set_xlim(atlas_pos.min(), atlas_pos.max()); top.set_ylim(0, 1)
    top.axvspan(*CUE_ZONE, color=ATLAS_CUE_NEUTRAL, lw=0)
    top.axvspan(*R1_ZONE, color=R1_COL, lw=0)
    top.axvspan(*R2_ZONE, color=R2_COL, lw=0)
    top.axvline(CUE_VISIBLE, color='0.25', ls='--', lw=0.5)
    top.set_xticks([]); top.set_yticks([])
    for sp in top.spines.values():
        sp.set_linewidth(0.4); sp.set_edgecolor('0.5')
    return top

def make_atlas_figure(save=True):
    plt.rcParams.update({'font.family': 'sans-serif', 'svg.fonttype': 'none'})
    NCOLS, NROWS = 7, 11
    L, R, TOP, BOT = 0.058, 0.895, 0.880, 0.078
    fig = plt.figure(figsize=(8.27, 11.69))                  # A4 portrait
    gs = GridSpec(NROWS, NCOLS, figure=fig, left=L, right=R, top=TOP, bottom=BOT,
                  wspace=0.14, hspace=0.78)
    extent = [atlas_pos.min(), atlas_pos.max(), len(atlas_sids), 0]
    cmap = plt.get_cmap(ATLAS_CMAP).copy(); cmap.set_bad('white')

    im = None
    for k, unit in enumerate(atlas_units):
        r, c = divmod(k, NCOLS)
        ax = fig.add_subplot(gs[r, c])
        M = _atlas_neuron_matrix(unit); peak = np.nanmax(M)
        norm = M / peak if np.isfinite(peak) and peak > 0 else M
        im = ax.imshow(norm, origin='upper', aspect='auto', cmap=cmap, vmin=0, vmax=1,
                       extent=extent, interpolation='nearest')
        for b in BOUNDARY_LINES:
            ax.axvline(b, color='0.45', ls=':', lw=0.4, alpha=0.7)
        if _atlas_cue_y:
            ax.axhline(_atlas_cue_y, color=ATLAS_CUEINTRO_COL, ls='--', lw=0.7)
        if _atlas_rev_y:
            ax.axhline(_atlas_rev_y, color=REVERSAL_COL, ls='--', lw=0.7)
        ax.set_xlim(atlas_pos.min(), atlas_pos.max()); ax.set_ylim(len(atlas_sids), 0)
        ax.set_yticks([])
        if r == NROWS - 1:
            ax.set_xticks([-100, 0, 100, 200])
            ax.tick_params(axis='x', labelsize=6, length=2, pad=1.5)
        else:
            ax.set_xticks([])
        for sp in ax.spines.values():
            sp.set_linewidth(0.5); sp.set_edgecolor('0.5')
        # own little track-zone strip above the panel; neuron label on its title
        top = _atlas_track_strip(ax)
        top.set_title(f'N{k+1} · {_atlas_fmt_hz(peak)} Hz', fontsize=6, pad=2.0,
                      color=ATLAS_HP_COL if k < ATLAS_N_HP else ATLAS_MPFC_COL)

    # colorbar
    cax = fig.add_axes([R + 0.018, BOT + 0.12, 0.014, TOP - BOT - 0.24])
    cb = fig.colorbar(im, cax=cax, ticks=[0, 0.5, 1])
    cb.ax.set_yticklabels(['0', '0.5', 'peak']); cb.ax.tick_params(labelsize=8)
    cb.set_label('Mean firing rate\n(per-neuron peak-normalised)', fontsize=9)
    cb.outline.set_linewidth(0.5)

    # titles / shared axis labels
    fig.suptitle('Single-neuron trackwise activation across sessions — all 77 units',
                 fontsize=13, fontweight='bold', x=L, ha='left', y=0.978)
    fig.text(L, 0.957, 'Mean firing rate per track position (pooled across cue 1 / cue 2); '
             'each panel peak-normalised, with its own track-zone strip. '
             'Panel title = neuron · peak rate.',
             fontsize=8.5, ha='left', color='0.25')
    fig.supxlabel('Track position [cm]', fontsize=11, y=0.046)
    fig.supylabel('Session  (top = earliest → bottom = latest recording)', fontsize=11, x=0.012)

    handles = [
        Patch(facecolor=ATLAS_HP_COL, label='Hippocampus (N1–20)'),
        Patch(facecolor=ATLAS_MPFC_COL, label='mPFC (N21–77)'),
        Patch(facecolor=ATLAS_CUE_NEUTRAL, label='Cue zone'),
        Patch(facecolor=R2_COL, edgecolor='0.4', label='Reward zone'),
        Line2D([0], [0], color='0.25', ls='--', lw=1.2, label='Cue visible'),
        Line2D([0], [0], color=ATLAS_CUEINTRO_COL, ls='--', lw=1.4, label='Cue introduced'),
        Line2D([0], [0], color=REVERSAL_COL, ls='--', lw=1.4, label='Reversal'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=8.5, frameon=False,
               bbox_to_anchor=(0.5, 0.004), columnspacing=1.5, handlelength=1.6)

    if save:
        stem = 'appendix_single_neuron_trackwise_atlas'
        fig.savefig(os.path.join(THESIS_FIG_DIR, stem + '.pdf'), dpi=300)
        fig.savefig(os.path.join(PREVIEW_DIR, stem + '.svg'), dpi=150)
        print('wrote', stem + '.pdf / .svg')
    return fig

make_atlas_figure();

# Fourth appendix figure — ensemble trackwise activation atlas

Companion to thesis **Figure 4** (assembly spatial-tuning heatmaps), generalised
from one example assembly to **all 23 ICA ensembles** and **pooled across cues**.
Every panel is a session × track-position heatmap of one ensemble's mean
projection (all trials, both cue 1 and cue 2). Unlike Figure 4 there is **no cue
split** — each panel shows the overall trackwise activation of a single ensemble
over learning, and carries **its own little track-zone strip** directly above it
(cue / reward zones + cue-visible marker), aligned to that panel's track-position
axis.

Ensemble projections are **signed**, so each panel is shown on a **diverging
colour scale anchored at 0** (red = positive, blue = negative) and is
**peak-normalised per ensemble** (±99th-percentile robust limit) so each
ensemble's spatial-tuning pattern is legible regardless of its magnitude; the
±colour-scale limit is printed in every panel title so the quantitative scale is
preserved. Sessions run top (earliest) → bottom (latest); the cue-introduction
(2024-11-27, teal) and reversal (2025-01-23, red) boundaries are dashed lines,
and faint dotted guides mark the cue-visible point and the cue / reward-zone
onsets within each heatmap. The whole atlas is laid out on a single **A4 portrait
page** (4 columns × 6 rows = 24 cells: 23 ensembles + a legend cell). Session set
= every ephys session minus the three quality/length-excluded sessions
(26 sessions), the same set as thesis Figure 4.

In [ ]:
# Fourth appendix figure — ensemble trackwise activation atlas (all 23 ensembles)
# Companion to thesis Figure 4 (assembly spatial-tuning heatmaps), generalised to
# every ICA ensemble and pooled across cues. Each small panel is a
# session x track-position heatmap of one ensemble's mean projection (all trials,
# both cue 1 and cue 2 -> NO cue split), with its OWN little track-zone strip
# directly above it (cue / reward zones + cue-visible marker), exactly like
# Figure 4's per-panel schematic. Projections are signed, so panels use a
# diverging scale anchored at 0 and are peak-normalised per ensemble (robust
# 99th-percentile limit) so each ensemble's spatial-tuning pattern is legible
# regardless of magnitude; the +/- colour-scale limit is printed in every panel
# title. One A4 portrait page, 4 cols x 6 rows = 24 cells (23 panels + legend).
# Reuses the config-cell constants (zones, colours, CUE_INTRO, REVERSAL_SID, ...).
# Saved without bbox_inches='tight' so the canvas stays exactly A4.
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable
from analytics_processing.sessions_from_nas_parsing import (
    sessionlist_fullfnames_from_args, fullfnames2snames)

# ensemble-atlas style constants (ENSA-prefixed so they never clobber the other
# figures' HP_COL / MPFC_COL / REVERSAL_DATE defined in the cells above)
ENSA_CUE_NEUTRAL = '#b9a7d6'         # cue zone in the strips (data are cue-pooled -> neutral)
ENSA_CUEINTRO_COL = '#1b7895'
ENSA_REV_DATE = REVERSAL_SID[:10]
ENSA_SMOOTH_SIG = (0.6, 3.0)         # (session, position) gaussian sigma, in bins
ENSA_CMAP = 'RdBu_r'                 # diverging, anchored at 0 (signed projections)
ENSA_TITLE_COL = '#222222'

def _ensa_nan_gaussian(M, sigma):
    '''NaN-aware 2D gaussian smoothing (normalised convolution).'''
    mask = np.isfinite(M).astype(float)
    num = gaussian_filter(np.where(mask > 0, M, 0.0), sigma=sigma, mode='nearest')
    den = gaussian_filter(mask, sigma=sigma, mode='nearest')
    out = np.divide(num, den, out=np.full_like(num, np.nan), where=den > 1e-6)
    out[mask == 0] = np.nan          # keep genuinely-missing bins blank
    return out

# --- session set (same as Figure 4 / the rest of A6): all minus the 3 excluded ---
_ENSA_EXCL = ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min',
              '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min',
              '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min']
_ensa_dirs = sessionlist_fullfnames_from_args([1100], [6], None, excl_session_names=_ENSA_EXCL)[0]
_ensa_session_names = fullfnames2snames(_ensa_dirs)

_ensa_tw = analytics.get_analytics('TrackwiseEnsembleProj', session_names=_ensa_session_names)
if 'session_id' not in _ensa_tw.columns:
    _ensa_tw = _ensa_tw.reset_index()
ensa_cols = [c for c in _ensa_tw.columns if str(c).startswith('Assembly')]
_ensa_tw['pos'] = pd.to_numeric(_ensa_tw['from_position_bin'], errors='coerce')
_ensa_tw = _ensa_tw.dropna(subset=['pos'])
_ensa_tw = _ensa_tw[(_ensa_tw['pos'] >= min_track) & (_ensa_tw['pos'] <= max_track)]
ensa_sids = sorted(_ensa_tw['session_id'].unique())
ensa_pos = np.sort(_ensa_tw['pos'].unique())
_ensa_grp = _ensa_tw.groupby(['session_id', 'pos'])          # group once (reused per ensemble)
print(f'{len(ensa_cols)} ensembles, {len(ensa_sids)} sessions, {len(ensa_pos)} position bins '
      f'[{ensa_pos.min():.0f}, {ensa_pos.max():.0f}] cm')

def _ensa_matrix(col):
    '''(session x position) mean-projection matrix, pooled over all cues, smoothed.'''
    m = _ensa_grp[col].mean().unstack('pos').reindex(index=ensa_sids, columns=ensa_pos)
    return _ensa_nan_gaussian(m.to_numpy(float), ENSA_SMOOTH_SIG)

_ensa_dates = [s[:10] for s in ensa_sids]
def _ensa_boundary_y(thresh):
    n = sum(d < thresh for d in _ensa_dates)
    return n if 0 < n < len(ensa_sids) else None
_ensa_cue_y = _ensa_boundary_y(CUE_INTRO)
_ensa_rev_y = _ensa_boundary_y(ENSA_REV_DATE)

def _ensa_fmt(v):
    if not np.isfinite(v):
        return 'n/a'
    return f'{v:.2f}' if v < 1 else (f'{v:.1f}' if v < 10 else f'{v:.0f}')

def _ensa_track_strip(ax):
    '''Thin per-panel track-zone schematic above the heatmap (cue + reward zones
    and the cue-visible marker), aligned to the panel's track-position axis.'''
    top = make_axes_locatable(ax).append_axes('top', size='15%', pad=0.03)
    top.set_xlim(ensa_pos.min(), ensa_pos.max()); top.set_ylim(0, 1)
    top.axvspan(*CUE_ZONE, color=ENSA_CUE_NEUTRAL, lw=0)
    top.axvspan(*R1_ZONE, color=R1_COL, lw=0)
    top.axvspan(*R2_ZONE, color=R2_COL, lw=0)
    top.axvline(CUE_VISIBLE, color='0.25', ls='--', lw=0.5)
    top.set_xticks([]); top.set_yticks([])
    for sp in top.spines.values():
        sp.set_linewidth(0.4); sp.set_edgecolor('0.5')
    return top

def make_ensemble_atlas_figure(save=True):
    plt.rcParams.update({'font.family': 'sans-serif', 'svg.fonttype': 'none'})
    NCOLS, NROWS = 4, 6
    L, R, TOP, BOT = 0.072, 0.895, 0.900, 0.072
    fig = plt.figure(figsize=(8.27, 11.69))                  # A4 portrait
    gs = GridSpec(NROWS, NCOLS, figure=fig, left=L, right=R, top=TOP, bottom=BOT,
                  wspace=0.16, hspace=0.62)
    extent = [ensa_pos.min(), ensa_pos.max(), len(ensa_sids), 0]
    cmap = plt.get_cmap(ENSA_CMAP).copy(); cmap.set_bad('0.85')

    # bottom-most occupied row per column, for x-tick placement
    last_row = {}
    for k in range(len(ensa_cols)):
        c = k % NCOLS; last_row[c] = max(last_row.get(c, -1), k // NCOLS)

    im = None
    for k, col in enumerate(ensa_cols):
        r, c = divmod(k, NCOLS)
        ax = fig.add_subplot(gs[r, c])
        M = _ensa_matrix(col)
        vpk = np.nanpercentile(np.abs(M), 99)                # robust per-panel scale limit
        if not np.isfinite(vpk) or vpk <= 0:
            vpk = np.nanmax(np.abs(M))
        norm = np.clip(M / vpk, -1, 1) if np.isfinite(vpk) and vpk > 0 else M
        im = ax.imshow(norm, origin='upper', aspect='auto', cmap=cmap, vmin=-1, vmax=1,
                       extent=extent, interpolation='nearest')
        for b in BOUNDARY_LINES:
            ax.axvline(b, color='0.35', ls=':', lw=0.4, alpha=0.55)
        if _ensa_cue_y:
            ax.axhline(_ensa_cue_y, color=ENSA_CUEINTRO_COL, ls='--', lw=0.8)
        if _ensa_rev_y:
            ax.axhline(_ensa_rev_y, color=REVERSAL_COL, ls='--', lw=0.8)
        ax.set_xlim(ensa_pos.min(), ensa_pos.max()); ax.set_ylim(len(ensa_sids), 0)
        ax.set_yticks([])
        if r == last_row[c]:
            ax.set_xticks([-100, 0, 100, 200])
            ax.tick_params(axis='x', labelsize=6.5, length=2, pad=1.5)
        else:
            ax.set_xticks([])
        for sp in ax.spines.values():
            sp.set_linewidth(0.5); sp.set_edgecolor('0.5')
        # own little track-zone strip above the panel; ensemble label on its title
        top = _ensa_track_strip(ax)
        top.set_title(f'Ens {k+1} · ±{_ensa_fmt(vpk)}', fontsize=7.5, pad=2.5,
                      color=ENSA_TITLE_COL, fontweight='bold')

    # colorbar (right margin)
    cax = fig.add_axes([R + 0.022, BOT + 0.14, 0.015, TOP - BOT - 0.30])
    cb = fig.colorbar(im, cax=cax, ticks=[-1, 0, 1])
    cb.ax.set_yticklabels(['−peak', '0', '+peak']); cb.ax.tick_params(labelsize=8)
    cb.set_label('Mean ensemble projection\n(per-panel peak-normalised, signed)', fontsize=9)
    cb.outline.set_linewidth(0.5)

    # legend in the spare bottom-right cell
    ax_leg = fig.add_subplot(gs[NROWS - 1, NCOLS - 1]); ax_leg.axis('off')
    handles = [
        Patch(facecolor=ENSA_CUE_NEUTRAL, label='Cue zone'),
        Patch(facecolor=R1_COL, label='Reward 1 zone'),
        Patch(facecolor=R2_COL, edgecolor='0.4', label='Reward 2 zone'),
        Line2D([0], [0], color='0.25', ls='--', lw=1.2, label='Cue visible'),
        Line2D([0], [0], color=ENSA_CUEINTRO_COL, ls='--', lw=1.4, label='Cue introduced'),
        Line2D([0], [0], color=REVERSAL_COL, ls='--', lw=1.4, label='Reversal'),
    ]
    ax_leg.legend(handles=handles, loc='center', fontsize=8.0, frameon=False,
                  handlelength=1.5, labelspacing=0.7, borderpad=0.2)

    # titles / shared axis labels
    fig.suptitle('Ensemble trackwise activation across sessions — all 23 ICA ensembles',
                 fontsize=13, fontweight='bold', x=L, ha='left', y=0.972)
    fig.text(L, 0.951, 'Mean ensemble projection per track position (pooled across cue 1 / cue 2); '
             'each panel peak-normalised (signed, diverging scale),\nwith its own track-zone strip. '
             'Panel title = ensemble · ±colour-scale limit (99th-percentile robust).',
             fontsize=8.5, ha='left', color='0.25', va='top')
    fig.supxlabel('Track position [cm]', fontsize=11, y=0.040)
    fig.supylabel('Session  (top = earliest → bottom = latest recording)', fontsize=11, x=0.014)

    if save:
        stem = 'appendix_ensemble_trackwise_atlas'
        fig.savefig(os.path.join(THESIS_FIG_DIR, stem + '.pdf'), dpi=300)
        fig.savefig(os.path.join(PREVIEW_DIR, stem + '.svg'), dpi=150)
        print('wrote', stem + '.pdf / .svg')
    return fig

make_ensemble_atlas_figure();